<a href="https://colab.research.google.com/github/vishaanthg08-source/TrendPulse/blob/part1-cleaning/part1_python_foundations/part1_python_foundations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TrendPulse - Part 1: Python Foundations
Core Python on YouTube trending data (US, GB, IN).


In [13]:
!git clone -q -b part1-cleaning https://github.com/vishaanthg08-source/trendpulse.git
%cd trendpulse/part1_python_foundations

/content/trendpulse/part1_python_foundations/trendpulse/part1_python_foundations/trendpulse/part1_python_foundations


In [14]:
import os
print(os.listdir("data"))

['.gitkeep', 'USvideos.csv', 'GBvideos.csv', 'categories.csv', 'INvideos.csv']


In [15]:
from datetime import datetime
import pandas as pd

# ---- Configuration ----
DATA_DIR = "data"                    # str
COUNTRIES = ["US", "GB", "IN"]       # list of str
CATEGORIES_FILE = f"{DATA_DIR}/categories.csv"   # str
VIRAL_VIEWS = 1_000_000              # int: views for Viral Hit
GOOD_LIKE_RATIO = 0.90               # float: good like ratio
DIVISIVE_LIKE_RATIO = 0.60           # float: below this = Divisive
TOP_N_CHANNELS = 15                  # int
RUN_START = datetime.now()           # datetime

# ---- Run header ----
print("=" * 45)
print("TrendPulse - Part 1 run")
print(f"Countries : {', '.join(COUNTRIES)}")
print(f"Started   : {RUN_START:%Y-%m-%d %H:%M:%S}")
print("=" * 45)

TrendPulse - Part 1 run
Countries : US, GB, IN
Started   : 2026-10-30 16:22:25


In [16]:
REQUIRED_COLS = ["video_id", "trending_date", "title", "channel_title", "category_id",
                 "publish_time", "views", "likes", "dislikes", "comment_count",
                 "comments_disabled", "ratings_disabled", "video_error_or_removed"]

def load_country(path, country, max_rows=None):
    df = pd.read_csv(path, nrows=max_rows)
    missing = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"{country}: missing columns {missing}")
    df["country"] = country
    return df.to_dict("records")

In [17]:
all_records = []
for c in COUNTRIES:
    rows = load_country(f"{DATA_DIR}/{c}videos.csv", c)
    print(f"{c}: {len(rows)} records loaded")
    all_records.extend(rows)

print("Total records:", len(all_records))

US: 40949 records loaded
GB: 38916 records loaded
IN: 37352 records loaded
Total records: 117217


In [18]:
cat_rows = pd.read_csv(CATEGORIES_FILE).to_dict("records")
category_names = {row["category_id"]: row["category_name"] for row in cat_rows}
print(len(category_names), "categories")
print(category_names)

32 categories
{1: 'Film & Animation', 2: 'Autos & Vehicles', 10: 'Music', 15: 'Pets & Animals', 17: 'Sports', 18: 'Short Movies', 19: 'Travel & Events', 20: 'Gaming', 21: 'Videoblogging', 22: 'People & Blogs', 23: 'Comedy', 24: 'Entertainment', 25: 'News & Politics', 26: 'Howto & Style', 27: 'Education', 28: 'Science & Technology', 29: 'Nonprofits & Activism', 30: 'Movies', 31: 'Anime/Animation', 32: 'Action/Adventure', 33: 'Classics', 34: 'Comedy', 35: 'Documentary', 36: 'Drama', 37: 'Family', 38: 'Foreign', 39: 'Horror', 40: 'Sci-Fi/Fantasy', 41: 'Thriller', 42: 'Shorts', 43: 'Shows', 44: 'Trailers'}


In [19]:
def to_int(value):
    return int(float(value))

def to_bool(value):
    text = str(value).strip().lower()
    if text in ("true", "1"):
        return True
    if text in ("false", "0"):
        return False
    raise ValueError(f"not a bool: {value!r}")

def parse_trending_date(text):
    return datetime.strptime(str(text).strip(), "%y.%d.%m").date()

def parse_publish_time(text):
    return datetime.strptime(str(text).strip(), "%Y-%m-%dT%H:%M:%S.%fZ")

In [20]:
INT_FIELDS = ["views", "likes", "dislikes", "comment_count"]
BOOL_FIELDS = ["comments_disabled", "ratings_disabled", "video_error_or_removed"]

failures = {}
for name in INT_FIELDS + BOOL_FIELDS + ["trending_date", "publish_time"]:
    failures[name] = 0

clean_records = []
dropped = 0

In [21]:
for rec in all_records:
    new = dict(rec)
    bad = False

    for f in INT_FIELDS:
        try:
            new[f] = to_int(rec[f])
        except (ValueError, TypeError):
            failures[f] += 1
            bad = True

    for f in BOOL_FIELDS:
        try:
            new[f] = to_bool(rec[f])
        except ValueError:
            failures[f] += 1
            new[f] = False

    try:
        new["trending_date"] = parse_trending_date(rec["trending_date"])
    except ValueError:
        failures["trending_date"] += 1
        bad = True

    try:
        new["publish_time"] = parse_publish_time(rec["publish_time"])
    except ValueError:
        failures["publish_time"] += 1
        bad = True

    if bad:
        dropped += 1
    else:
        clean_records.append(new)

In [22]:
print("Before :", len(all_records))
print("After  :", len(clean_records))
print("Dropped:", dropped)
print("Failures per field:")
for name in failures:
    print(name, "=", failures[name])

Before : 117217
After  : 117217
Dropped: 0
Failures per field:
views = 0
likes = 0
dislikes = 0
comment_count = 0
comments_disabled = 0
ratings_disabled = 0
video_error_or_removed = 0
trending_date = 0
publish_time = 0


## Cleaning decisions
- views, likes, dislikes, comment_count: converted to int. A bad value drops the record, because replacing it with 0 would distort averages.
- True/False flags: converted to bool. A bad value is replaced with False.
- trending_date is stored as yy.dd.mm (17.14.11 = 14 Nov 2017), parsed with datetime.strptime.
- publish_time is ISO format, parsed with datetime.strptime.
- try/except wraps every conversion, so one bad value cannot crash the run.

All 117,217 records parsed successfully, so no record was dropped and every failure count is 0. The try/except handling is still in place for messy data.